In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# --- Config ---
RMSD_FEATURES = [
    'RMSD_chA_aft_chB_align_af2_af3', 'RMSD_chA_aft_chB_align_af2_boltz1',
    'RMSD_chA_aft_chB_align_af2_colab', 'RMSD_chA_aft_chB_align_boltz1_af3',
    'RMSD_chA_aft_chB_align_colab_af3', 'RMSD_chA_aft_chB_align_colab_boltz1',
    'RMSD_chA_aft_chB_align_input_af2', 'RMSD_chA_aft_chB_align_input_af3',
    'RMSD_chA_aft_chB_align_input_boltz1', 'RMSD_chA_aft_chB_align_input_colab',
    'RMSD_complex_af2_af3', 'RMSD_complex_af2_boltz1',
    'RMSD_complex_af2_colab', 'RMSD_complex_boltz1_af3',
    'RMSD_complex_colab_af3', 'RMSD_complex_colab_boltz1',
    'RMSD_complex_input_af2', 'RMSD_complex_input_af3',
    'RMSD_complex_input_boltz1', 'RMSD_complex_input_colab',
]

REFERENCE_FEAT = 'af3_ipSAE_min'

# --- New simplified color scheme ---
color_map = {
    'input': '#9E9E9E',   # light grey
    'cross': '#036c5f',   # dark grey
}

def _model_color(feat_name: str) -> str:
    """Classify features into 'input' vs 'cross-model'."""
    if 'input' in feat_name:
        return color_map['input']
    return color_map['cross']

def best_ap(vals, y):
    """Return the best AP (higher / lower direction) and the direction used."""
    vals = pd.to_numeric(vals, errors='coerce').fillna(0)
    try:
        ap_high = average_precision_score(y, vals)
        ap_low  = average_precision_score(y, -vals)
    except ValueError:
        return np.nan, False
    if ap_high >= ap_low:
        return ap_high, False   # higher is better
    return ap_low, True         # lower is better

# --- Compute APs ---
y = df['binder'].values

rows = []
for feat in RMSD_FEATURES:
    if feat not in df.columns:
        print(f"[WARN] {feat} not found – skipping")
        continue
    ap, lower = best_ap(df[feat], y)
    rows.append({'feature': feat, 'ap': ap, 'lower_is_better': lower})

df_rmsd = pd.DataFrame(rows).dropna(subset=['ap'])
df_rmsd = df_rmsd.sort_values('ap', ascending=True).reset_index(drop=True)

# Reference AP
ref_ap = np.nan
if REFERENCE_FEAT in df.columns:
    ref_ap, _ = best_ap(df[REFERENCE_FEAT], y)
else:
    print(f"[WARN] Reference feature '{REFERENCE_FEAT}' not found in dataframe.")

# --- Plot ---
fig, ax = plt.subplots(dpi=300, figsize=(7.5, 6.5))

bar_colors  = [_model_color(f) for f in df_rmsd['feature']]
arrow_heads = []   # track features where lower_is_better for annotation

for i, row in df_rmsd.iterrows():
    ax.barh(i, row['ap'], color=bar_colors[i], edgecolor='black', height=0.65)
    # Value label
    ax.text(row['ap'] + 0.005, i, f"{row['ap']:.3f}",
            va='center', fontsize=8.5)
    # Small left-arrow marker when lower-is-better
    if row['lower_is_better']:
        ax.text(row['ap'] + 0.005 + 0.035, i, '↓', va='center',
                fontsize=8, color='dimgray')

# --- Reference line ---
if not np.isnan(ref_ap):
    ax.axvline(ref_ap, color='black', linewidth=1.6,
               linestyle='--', zorder=5, label=f'{REFERENCE_FEAT}  (AP={ref_ap:.3f})')
    ax.text(ref_ap + 0.004, len(df_rmsd) - 0.3,
            f'{REFERENCE_FEAT}\nAP={ref_ap:.3f}',
            fontsize=8, va='top', color='black',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.7, ec='none'))

# --- Axes & labels ---
ax.set_yticks(range(len(df_rmsd)))
ax.set_yticklabels(df_rmsd['feature'], fontsize=8.5)
ax.set_xlabel('Average Precision (AP)')
ax.set_xlim(0, df_rmsd['ap'].max() + 0.12)
ax.set_title('RMSD Feature AP vs. AF3_ipSAE_min', loc='left', fontweight='bold')

legend_patches = [
    mpatches.Patch(facecolor=color_map['input'], edgecolor='black', label='RMSD to input'),
    mpatches.Patch(facecolor=color_map['cross'], edgecolor='black', label='RMSD cross model'),
]

ref_line = plt.Line2D([0], [0], color='black', linewidth=1.6,
                      linestyle='--', label=f'Reference: {REFERENCE_FEAT}')

ax.legend(handles=legend_patches + [ref_line],
          loc='lower right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
#plt.savefig("rmsd_ap_vs_reference.png", bbox_inches='tight')
plt.show()


# --- Interaction features: af3_ipSAE_min * (1 / RMSD) ---

rows_inter = []

if REFERENCE_FEAT not in df.columns:
    print(f"[WARN] Cannot compute interaction features: '{REFERENCE_FEAT}' missing.")
else:
    ref_vals = pd.to_numeric(df[REFERENCE_FEAT], errors='coerce')

    for feat in RMSD_FEATURES:
        if feat not in df.columns:
            continue

        rmsd_vals = pd.to_numeric(df[feat], errors='coerce')

        # Avoid division by zero
        inv_rmsd = 1.0 / rmsd_vals.replace(0, np.nan)

        interaction = ref_vals * inv_rmsd

        ap, lower = best_ap(interaction, y)

        rows_inter.append({
            'feature': f"{REFERENCE_FEAT} * 1/{feat}",
            'ap': ap,
            'lower_is_better': lower
        })

df_inter = pd.DataFrame(rows_inter).dropna(subset=['ap'])
df_inter = df_inter.sort_values('ap', ascending=True).reset_index(drop=True)

# --- Plot interaction features ---
fig, ax = plt.subplots(dpi=300, figsize=(7.5, 6.5))

bar_colors = [_model_color(f) for f in df_inter['feature']]

for i, row in df_inter.iterrows():
    ax.barh(i, row['ap'], color=bar_colors[i], edgecolor='black', height=0.65)

    # Value label
    ax.text(row['ap'] + 0.005, i, f"{row['ap']:.3f}",
            va='center', fontsize=8.5)

    # Arrow if lower is better
    if row['lower_is_better']:
        ax.text(row['ap'] + 0.005 + 0.035, i, '↓',
                va='center', fontsize=8, color='dimgray')

# --- Reference line (same as before) ---
if not np.isnan(ref_ap):
    ax.axvline(ref_ap, color='black', linewidth=1.6,
               linestyle='--', zorder=5,
               label=f'{REFERENCE_FEAT}  (AP={ref_ap:.3f})')

# --- Axes ---
ax.set_yticks(range(len(df_inter)))
ax.set_yticklabels(df_inter['feature'], fontsize=7.5)
ax.set_xlabel('Average Precision (AP)')
ax.set_xlim(0, df_inter['ap'].max() + 0.12)
ax.set_title('Interaction: AF3_ipSAE_min × (1 / RMSD)', loc='left', fontweight='bold')

legend_patches = [
    mpatches.Patch(facecolor=color_map['input'], edgecolor='black', label='RMSD to input'),
    mpatches.Patch(facecolor=color_map['cross'], edgecolor='black', label='RMSD cross model'),
]

ref_line = plt.Line2D([0], [0], color='black', linewidth=1.6,
                      linestyle='--', label=f'Reference: {REFERENCE_FEAT}')

ax.legend(handles=legend_patches + [ref_line],
          loc='lower right', fontsize=8.5, framealpha=0.9)

plt.tight_layout()
plt.savefig("interaction_ap_vs_reference.png", bbox_inches='tight')
plt.show()